In [1]:
# %% [markdown]
# ### 1. تجهيز المكتبات الخارجية والمسارات البيئية

# %%
import os
import sys
import shutil

offline_packages_path = "/home/jovyan/work/storage/packages"
os.makedirs(offline_packages_path, exist_ok=True)

if not os.path.exists(os.path.join(offline_packages_path, "psycopg2")):
    print("📦 جاري تثبيت psycopg2 للمرة الأولى...")
    !pip install --target={offline_packages_path} psycopg2-binary --quiet

if offline_packages_path not in sys.path:
    sys.path.insert(0, offline_packages_path)

import psycopg2
print("✅ تم إعداد بيئة المكتبات بنجاح!")

# %% [markdown]
# ### 2. بناء جلسة Spark موحدة وشاملة الإعدادات

# %%
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, when, expr, window, avg, sum
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, BooleanType

spark = SparkSession.builder \
    .appName("SmartHome-Energy-Streaming") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0,org.postgresql:postgresql:42.6.0") \
    .config("spark.sql.streaming.minBatchesToRetain", "30") \
    .getOrCreate()

for q in spark.streams.active:
    q.stop()

print("⚙️ تم تشغيل جلسة Spark الموحدة بنجاح!")

# %% [markdown]
# ### 3. بناء الـ Schema والربط بكافكا

# %%
schema = StructType([
    StructField("timestamp", StringType(), True),
    StructField("house_type", StringType(), True),
    StructField("currency", StringType(), True),
    StructField("zone", StringType(), True),
    StructField("device_id", StringType(), True),
    StructField("device_type", StringType(), True),
    StructField("is_room_occupied", BooleanType(), True),
    StructField("power_consumption_watts", DoubleType(), True),
    StructField("status", StringType(), True)
])

kafka_stream_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "smarthome-kafka:29092") \
    .option("subscribe", "energy_events") \
    .option("startingOffsets", "earliest") \
    .option("failOnDataLoss", "false") \
    .load()

print("📡 تم الاتصال بـ Kafka وبدء الاستماع...")

# %% [markdown]
# ### 4. معالجة البيانات وتطبيق الفلاتر والنوافذ الزمنية

# %%
parsed_stream_df = kafka_stream_df.selectExpr("CAST(value AS STRING) as json_str") \
    .select(from_json(col("json_str"), schema).alias("data")) \
    .select("data.*") \
    .withColumn("timestamp", col("timestamp").cast("timestamp"))

watermarked_df = parsed_stream_df.withWatermark("timestamp", "20 hours")

context_clean_df = watermarked_df \
    .filter(
        (col("power_consumption_watts").isNotNull()) & 
        (~col("status").isin("LOST_SIGNAL", "SENSOR_FAULT", "ERROR")) &
        (col("power_consumption_watts") >= 0) &
        (
            ((col("device_type") == "AC") & (col("power_consumption_watts") <= 6000)) |
            ((col("device_type") == "Oven") & (col("power_consumption_watts") <= 4000)) |
            ((col("device_type") == "Washing_Machine") & (col("power_consumption_watts") <= 3000)) |
            ((col("device_type").isin("Lighting", "Smart_Bulb", "TV")) & (col("power_consumption_watts") <= 500)) |
            (~col("device_type").isin("AC", "Oven", "Washing_Machine", "Lighting", "Smart_Bulb", "TV") & (col("power_consumption_watts") <= 2000))
        )
    ) \
    .dropDuplicates(["timestamp", "device_id"])

advanced_processed_df = context_clean_df \
    .groupBy(
        window(col("timestamp"), "5 minutes", "1 minute"),
        col("zone"),
        col("device_type")
    ) \
    .agg(
        avg("power_consumption_watts").alias("avg_power_watts"),
        sum("power_consumption_watts").alias("total_power_watts")
    ) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("zone"),
        col("device_type"),
        col("avg_power_watts"),
        col("total_power_watts")
    )

print("🚀 خط معالجة البيانات جاهز للإطلاق.")

# %% [markdown]
# ### 5. دالة الـ Upsert والتنظيف الدوري لـ Postgres

# %%
def write_to_postgres_only(batch_df, batch_id):
    if batch_df.rdd.isEmpty():
        return

    batch_df.cache()
    row_count = batch_df.count()
    
    if row_count > 0:
        print(f"📥 [Batch {batch_id}] جاري معالجة وضخ {row_count} سجل...")
        try:
            conn = psycopg2.connect(
                host="smarthome-postgres", database="smarthome_energy", 
                user="smarthome_user", password="smarthome_password", port="5432"
            )
            cursor = conn.cursor()
            records = batch_df.collect()
            
            upsert_query = """
                INSERT INTO spark_windowed_energy (zone, device_type, avg_power_watts, total_power_watts, window_start, window_end)
                VALUES (%s, %s, %s, %s, %s, %s)
                ON CONFLICT (zone, device_type, window_end) 
                DO UPDATE SET 
                    avg_power_watts = EXCLUDED.avg_power_watts,
                    total_power_watts = EXCLUDED.total_power_watts,
                    window_start = EXCLUDED.window_start;
            """
            data_to_insert = [
                (r['zone'], r['device_type'], float(r['avg_power_watts']), float(r['total_power_watts']), r['window_start'], r['window_end'])
                for r in records
            ]
            cursor.executemany(upsert_query, data_to_insert)
            conn.commit()
            cursor.close()
            conn.close()
            print("   🔹 [Postgres] تم الـ Upsert بنجاح.")
        except Exception as e:
            print(f"   ❌ فشل في Postgres: {e}")
        
        # التنظيف التلقائي كل 5 دفعات للأرشيف اللحظي القديم
        if batch_id % 5 == 0:
            try:
                conn = psycopg2.connect(host="smarthome-postgres", database="smarthome_energy", user="smarthome_user", password="smarthome_password", port="5432")
                cursor = conn.cursor()
                cursor.execute("""
                    DELETE FROM spark_windowed_energy 
                    WHERE window_end < (SELECT MAX(window_end) FROM spark_windowed_energy) - INTERVAL '10 minutes';
                """)
                conn.commit()
                cursor.close()
                conn.close()
                print("   🧹 [Auto-Purge] تم تنظيف البيانات التاريخية القديمة اللحظية.")
            except Exception as e:
                print(f"   ⚠️ تنبيه الحذف التلقائي: {e}")
                
    batch_df.unpersist()

# %% [markdown]
# ### 6. تشغيل خط الأنابيب اللحظي وإبقاء الاتصال مستقراً

# %%
print("🚀 إطلاق تيار البث اللحظي للبوستغرس...")

query = (advanced_processed_df.writeStream
    .foreachBatch(write_to_postgres_only)
    .outputMode("update")
    .trigger(processingTime='10 seconds') 
    .option("checkpointLocation", "/home/jovyan/work/storage/check_spark_to")
    .start())

query.awaitTermination()

✅ تم إعداد بيئة المكتبات بنجاح!
⚙️ تم تشغيل جلسة Spark الموحدة بنجاح!
📡 تم الاتصال بـ Kafka وبدء الاستماع...
🚀 خط معالجة البيانات جاهز للإطلاق.
🚀 إطلاق تيار البث اللحظي للبوستغرس...
📥 [Batch 0] جاري معالجة وضخ 165 سجل...
   🔹 [Postgres] تم الـ Upsert بنجاح.
   🧹 [Auto-Purge] تم تنظيف البيانات التاريخية القديمة اللحظية.
📥 [Batch 1] جاري معالجة وضخ 66 سجل...
   🔹 [Postgres] تم الـ Upsert بنجاح.


ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: reentrant call inside <_io.BufferedReader name=52>

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving
ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip

Py4JError: An error occurred while calling o155.awaitTermination